### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [1]:
import os
import sys
from langchain_community.document_loaders import ConfluenceLoader
from atlassian import Confluence

sys.path.append(os.path.abspath(".."))

/Users/manishkumar/project/personnel/RAGBuilder/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Define constants in constants.py and import them here
from data.constants import token, username, url, url_extended

In [3]:

# Initialize Confluence client
confluence = Confluence(
    url=url,
    username=username,
    password=token,
    cloud=True
)

# Get all spaces from Confluence
spaces = confluence.get_all_spaces(start=0, limit=100)
confluence_documents = []

# Loop through spaces
for space in spaces.get("results", []):
    space_key = space["key"]
    print(f"Loading space: {space_key}")

    # Initialize ConfluenceLoader for the current space
    loader = ConfluenceLoader(
        url=url_extended,
        username=username,
        api_key=token,
        space_key=space_key
    )

    # Load documents from the current space
    docs = loader.load()
    
    # Append loaded documents to the main list
    confluence_documents.extend(docs)

print(f"Total docs loaded: {len(confluence_documents)}")


Loading space: ~5e70b60e8ae93a0c5a78b780
Loading space: ~5570580a7ceda9e2bf4c2fa36ec76b019dc54a
Loading space: MFS
Total docs loaded: 9


In [4]:
print(confluence_documents)

[Document(metadata={'title': 'Overview', 'id': '589917', 'source': 'https://manishsahay.atlassian.net/wiki/spaces/~5e70b60e8ae93a0c5a78b780/overview', 'when': '2026-03-09T14:26:01.270Z'}, page_content='Say hello to your colleagues who want to know your name, pronouns, role, team and location (or if you\'re remote). 📄 Recent content that I\'ve worked on 5 5 titles 🖐 Get in touch ✉️ Insert your email here 💼 Insert your LinkedIn URL here 🔗 Insert your Twitter handle here 👤 Insert your Medium profile here End with a bang! Some options are: "I am so grateful to be here at <Insert company name> and very excited to get started!" or "Looking forward to meeting all of you!" or "Can\'t wait to get to know all of you!"'), Document(metadata={'title': 'Overview', 'id': '163939', 'source': 'https://manishsahay.atlassian.net/wiki/spaces/~5570580a7ceda9e2bf4c2fa36ec76b019dc54a/overview', 'when': '2026-03-09T11:53:24.827Z'}, page_content='Say hello to your colleagues who want to know your name, pronoun

In [5]:
from src.utility import split_documents

chunks=split_documents(confluence_documents)
chunks

Split 9 documents into 81 chunks

Example chunk:
Content: Say hello to your colleagues who want to know your name, pronouns, role, team and location (or if you're remote). 📄 Recent content that I've worked on 5 5 titles 🖐 Get in touch ✉️ Insert your email he...
Metadata: {'title': 'Overview', 'id': '589917', 'source': 'https://manishsahay.atlassian.net/wiki/spaces/~5e70b60e8ae93a0c5a78b780/overview', 'when': '2026-03-09T14:26:01.270Z'}


[Document(metadata={'title': 'Overview', 'id': '589917', 'source': 'https://manishsahay.atlassian.net/wiki/spaces/~5e70b60e8ae93a0c5a78b780/overview', 'when': '2026-03-09T14:26:01.270Z'}, page_content='Say hello to your colleagues who want to know your name, pronouns, role, team and location (or if you\'re remote). 📄 Recent content that I\'ve worked on 5 5 titles 🖐 Get in touch ✉️ Insert your email here 💼 Insert your LinkedIn URL here 🔗 Insert your Twitter handle here 👤 Insert your Medium profile here End with a bang! Some options are: "I am so grateful to be here at <Insert company name> and very excited to get started!" or "Looking forward to meeting all of you!" or "Can\'t wait to get to know all of you!"'),
 Document(metadata={'title': 'Overview', 'id': '163939', 'source': 'https://manishsahay.atlassian.net/wiki/spaces/~5570580a7ceda9e2bf4c2fa36ec76b019dc54a/overview', 'when': '2026-03-09T11:53:24.827Z'}, page_content='Say hello to your colleagues who want to know your name, pronou

In [6]:
from src.embedding_manager import EmbeddingManager

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1739.28it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


In [7]:
### embedding And vectorStoreDB
from src.vector_store import VectorStore

vectorstore=VectorStore()
vectorstore


Vector store initialized. Collection: confluence_documents
Existing documents in collection: 3506


In [8]:
chunks

[Document(metadata={'title': 'Overview', 'id': '589917', 'source': 'https://manishsahay.atlassian.net/wiki/spaces/~5e70b60e8ae93a0c5a78b780/overview', 'when': '2026-03-09T14:26:01.270Z'}, page_content='Say hello to your colleagues who want to know your name, pronouns, role, team and location (or if you\'re remote). 📄 Recent content that I\'ve worked on 5 5 titles 🖐 Get in touch ✉️ Insert your email here 💼 Insert your LinkedIn URL here 🔗 Insert your Twitter handle here 👤 Insert your Medium profile here End with a bang! Some options are: "I am so grateful to be here at <Insert company name> and very excited to get started!" or "Looking forward to meeting all of you!" or "Can\'t wait to get to know all of you!"'),
 Document(metadata={'title': 'Overview', 'id': '163939', 'source': 'https://manishsahay.atlassian.net/wiki/spaces/~5570580a7ceda9e2bf4c2fa36ec76b019dc54a/overview', 'when': '2026-03-09T11:53:24.827Z'}, page_content='Say hello to your colleagues who want to know your name, pronou

In [9]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 81 texts...


Batches: 100%|██████████| 3/3 [00:00<00:00,  5.00it/s]

Generated embeddings with shape: (81, 384)
Adding 81 documents to vector store...
Successfully added 81 documents to vector store
Total documents in collection: 3587


In [10]:
### Retriever Pipeline From VectorStore

In [11]:
from src.rag_retriever import RAGRetriever

rag_retriever=RAGRetriever(vectorstore,embedding_manager)
rag_retriever


In [12]:
rag_retriever.retrieve("Lcqmc: A large-scale chinese question matching corpus")

Retrieving documents for query: 'Lcqmc: A large-scale chinese question matching corpus'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.31it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_0f724c68_382',
  'content': '[61] Liu, Xin, Qingcai Chen, Chong Deng, Huajun Zeng, Jing Chen, Do ngfang Li,\nand Buzhou Tang. ”Lcqmc: A large-scale chinese question matching corpus.” In\nProceedings of the 27th international conference on computatio nal linguistics, pp.\n1952-1962. 2018.\n[62] Yang, Yinfei, Yuan Zhang, Chris Tar, and Jason Baldridge. ”PA W S-X: A\ncross-lingual adversarial dataset for paraphrase identiﬁcation .” arXiv preprint\narXiv:1908.11828 (2019).\n[63] Cer, Daniel, Mona Diab, Eneko Agirre, Inigo Lopez-Gazpio, and L ucia Specia.\n”Semeval-2017 task 1: Semantic textual similarity-multilingual and c ross-lingual\nfocused evaluation.” arXiv preprint arXiv:1708.00055 (2017).\n[64] Tri Nguyen, Mir Rosenberg, Xia Song, Jianfeng Gao, Saurabh T iwary, Rangan\nMajumder, and Li Deng. 2016. MS MARCO: A human generated mach ine read-\ning comprehension dataset. In Proceedings of the Workshop on Co gnitive Com-\nputation: Integrating neural and symbolic approaches

In [13]:
from src.grok_llm import GroqLLM
from data.constants import groq_api_key

# Initialize Groq LLM (you'll need to set GROQ_API_KEY environment variable)
try:
    groq_llm = GroqLLM(api_key=groq_api_key)
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

    

Initialized Groq LLM with model: gemma2-9b-it
Groq LLM initialized successfully!


In [14]:
rag_retriever.retrieve("Lcqmc: A large-scale chinese question matching corpus")

Retrieving documents for query: 'Lcqmc: A large-scale chinese question matching corpus'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 120.24it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_0f724c68_382',
  'content': '[61] Liu, Xin, Qingcai Chen, Chong Deng, Huajun Zeng, Jing Chen, Do ngfang Li,\nand Buzhou Tang. ”Lcqmc: A large-scale chinese question matching corpus.” In\nProceedings of the 27th international conference on computatio nal linguistics, pp.\n1952-1962. 2018.\n[62] Yang, Yinfei, Yuan Zhang, Chris Tar, and Jason Baldridge. ”PA W S-X: A\ncross-lingual adversarial dataset for paraphrase identiﬁcation .” arXiv preprint\narXiv:1908.11828 (2019).\n[63] Cer, Daniel, Mona Diab, Eneko Agirre, Inigo Lopez-Gazpio, and L ucia Specia.\n”Semeval-2017 task 1: Semantic textual similarity-multilingual and c ross-lingual\nfocused evaluation.” arXiv preprint arXiv:1708.00055 (2017).\n[64] Tri Nguyen, Mir Rosenberg, Xia Song, Jianfeng Gao, Saurabh T iwary, Rangan\nMajumder, and Li Deng. 2016. MS MARCO: A human generated mach ine read-\ning comprehension dataset. In Proceedings of the Workshop on Co gnitive Com-\nputation: Integrating neural and symbolic approaches

In [15]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from src.rag_pipeline import RAGPipeline
from langchain_groq import ChatGroq

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.1-8b-instant",temperature=0.1,max_tokens=1024)

# Example usage:
adv_rag = RAGPipeline(rag_retriever, llm)


In [16]:
answer=adv_rag.query_simple("Lcqmc: A large-scale chinese question matching corpus")
print(answer)

Retrieving documents for query: 'Lcqmc: A large-scale chinese question matching corpus'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 117.29it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Lcqmc: A large-scale Chinese question matching corpus.


In [17]:

result = adv_rag.query_advanced("Lcqmc: A large-scale chinese question matching corpus", top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'Lcqmc: A large-scale chinese question matching corpus'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 109.79it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


Answer: Lcqmc: A large-scale Chinese question matching corpus, was introduced by Liu et al. in 2018.
Sources: [{'source': 'embeddings.pdf', 'page': 20, 'score': 0.3202354311943054, 'preview': '[61] Liu, Xin, Qingcai Chen, Chong Deng, Huajun Zeng, Jing Chen, Do ngfang Li,\nand Buzhou Tang. ”Lcqmc: A large-scale chinese question matching corpus.” In\nProceedings of the 27th international conference on computatio nal linguistics, pp.\n1952-1962. 2018.\n[62] Yang, Yinfei, Yuan Zhang, Chris Tar, a...'}, {'source': 'embeddings.pdf', 'page': 20, 'score': 0.3202354311943054, 'preview': '[61] Liu, Xin, Qingcai Chen, Chong Deng, Huajun Zeng, Jing Chen, Do ngfang Li,\nand Buzhou Tang. ”Lcqmc: A large-scale chinese question matching corpus.” In\nProceedings of the 27th international conference on computatio nal linguistics, pp.\n1952-1962. 2018.\n[62] Yang, Yinfei, Yuan Zhang, Chris Tar, a...'}, {'source': 'embeddings.pdf', 'page': 20, 'score': 0.3202354311943054, 'preview': '[61] Liu, Xin, Qingca

In [18]:

result = adv_rag.query_formatted("Lcqmc: A large-scale chinese question matching corpus", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'Lcqmc: A large-scale chinese question matching corpus'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 102.86it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
[61] Liu, Xin, Qingcai Chen, Chong Deng, Huajun Zeng, Jing Chen, Do ngfang Li,
and Buzhou Tang. ”Lcqmc: A large-scale chinese question matching corpus.” In
Proceedings of the 27th international conference on computatio nal linguistics, pp.
1952-1962. 

2018.
[62] Yang, Yinfei, Yuan Zhang, Chris Tar, and Jason Baldridge. ”PA W S-X: A
cross-lingual adversarial dataset for paraphrase identiﬁcation .” arXiv preprint
arXiv:1908.11828 (2019).
[63] Cer, Daniel, Mona Diab, Eneko Agirre, Inigo Lopez-Gazpio, and L ucia Specia.
”Semeval-2017 task 1: Semantic textual similarity-multilingual and c ross-lingual
focused evaluation.” arXiv preprint arXiv:1708.00055 (2017).
[64] Tri Nguyen, Mir Rosenberg, Xia Song, Jianfeng Gao, Saurabh T iwary, Rangan
Majumder, and Li Deng. 2016. MS MARCO: A human generated mach ine read-
ing comprehension dataset. In Proceedings of the Workshop on Co gnitive Com-
putation: Integrating neural and symbolic approaches 2016 co-lo cated with the

[61] Liu, Xin, Qingcai Chen, Chong Deng, Huajun Zeng, Jing Chen, Do ngfang Li,
and Buzhou Tang. ”Lcqmc: A large-scale chinese question matching corpus.” In
Proceedings of the 27th international conference on computatio nal linguistics, pp.
1952-1962. 2018.
[62] Yang, Yinfei, Yu